<h1>Building Makemore </h1>

In [261]:
import torch 
import torch.nn.functional as F

import matplotlib.pyplot as plt 
%matplotlib inline 

words = open('names.txt', 'r').read().splitlines()

In [262]:
# The bigram counts the frequencies of consecutive character pairs 
b = {}
for w in words:
    chars = ['.'] + list(w) + ['.']
    for char1, char2 in zip(chars, chars[1:]):
        bigram = (char1, char2)
        b[bigram] = b.get(bigram, 0) + 1
        # print(char1, char2)

# -----------------------------------------------------------------------------

# Sorting the pair:frequency tuples
# print(sorted(b.items(), key = lambda kv : -kv[1]))

In [263]:
# Initialize 28x28 array 
N = torch.zeros((27, 27), dtype = torch.int32)

# Create sorted list of characters a, ..., z, <S>, <E> corresponding to 1, ..., 27; and one for vice-versa 
chars = sorted(list(set(''.join(words))))
stoi = {s:i + 1 for i,s in enumerate(chars)} ; stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

for w in words:
    # sandwiching the word's characters between dots
    chs = ['.'] + list(w) + ['.']

    # isolate, and index through each consecutive character pair 
    for ch1, ch2 in zip(chs, chs[1:]):
        # find the corresponding integers of these two characters
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        
        # append the value stored in the cell (ix1, ix2) by 1
        N[ix1, ix2] += 1

<h1>Correlational Heatmap of Consecutive Character Pairings </h1>

<img src="output.png" style="width: 60%; height: auto;">

Generated by:
<small>
```python
plt.figure(figsize = (16, 16))
plt.imshow(N, cmap = 'Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha = 'center', va = 'bottom', color = 'gray')
        plt.text(j, i, N[i, j].item(), ha = 'center', va = 'top', color = 'gray')
plt.axis('off');
```
</small>


In [264]:
# P = N.float()
P = (N+1).float() # padded by 1 for smoothening, to avoid inconvenient cost/loss amounts such as +/- inf
P /= P.sum(1, keepdim = True)
g = torch.Generator().manual_seed(2147483647)

# -----------------------------------------------------------------------------

for i in range(5):
    out = []
    ix = 0 
    while True: 
        # p = N[ix].float()
        # p = p / p.sum()
        # p = torch.ones(27) / 27.0
        p = P[ix]
        ix = torch.multinomial(p, num_samples = 1, replacement = True, generator = g).item()
        out.append(itos[ix])
        if ix == 0:
            break 
    print(''.join(out))

junide.
janasah.
p.
cony.
a.


In [265]:
log_likelihood = 0.0 
n = 0 

# for w in words[:3]:
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1

        # print(f'{ch1}{ch2}: {prob:.4f}, {logprob:.4f}')

# -----------------------------------------------------------------------------
# Regular, Negative, and Normalized Log Likelihood are tantamount to a cost/loss function

print(f'{log_likelihood=}')
negative_log_likelihood = -log_likelihood
print(f'{negative_log_likelihood=}')
print(f'Normalized Log Likelihood: {negative_log_likelihood / n}')

log_likelihood=tensor(-559951.5625)
negative_log_likelihood=tensor(559951.5625)
Normalized Log Likelihood: 2.4543561935424805


<h1> Demonstrating the Effects of Smoothening on Loss </h1>

Code: 
<small>
```python
log_likelihood = 0.0 
n = 0 
for w in ['pranavq']:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1
print(f'{log_likelihood=}')
negative_log_likelihood = -log_likelihood
print(f'{negative_log_likelihood=}')
print(f'Normalized Log Likelihood: {negative_log_likelihood / n}')
```
</small>

<h1> Motivating a Loss Function for Gradient Descent</h1>

Code: 
<small>
```python
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator = g)
xenc = F.one_hot(xs, num_classes = 27).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts 
# Softmax:
counts = logits.exp() # counts, equivalent to N 
probs = counts / counts.sum(1, keepdim = True) # probabilities for next character 
nlls = torch.zeros(5)
for i in range(5):
    x = xs[i].item() # input character index 
    y = ys[i].item() # label character index 
    print('==========')
    print(f'bigram example {i + 1}: {itos[x]}{itos[y]} (indexes {x}, {y})')
    print(f'input to the neural net: {x}')
    print(f'output probabilities from the neural net: \n{probs[i]}')
    print(f'label (actual next character): {y}')
    p = probs[i, y]
    print(f'probability assigned by the net to the correct character: {p.item()}')
    logp = torch.log(p)
    print(f'log likelihood: {logp.item()}')
    nll = -logp 
    print(f'negative log likelihood: {nll.item()}')
    nlls[i] = nll 
print('==========')
print(f'average negative lot likelihood, i.e. loss = {nlls.mean().item()}')
```
</small>

Outputs:
<small>
```
bigram example 1: .e (indexes 0, 5)
input to the neural net: 0
output probabilities from the neural net: 
tensor([0.0607, 0.0100, 0.0123, 0.0042, 0.0168, 0.0123, 0.0027, 0.0232, 0.0137,
        0.0313, 0.0079, 0.0278, 0.0091, 0.0082, 0.0500, 0.2378, 0.0603, 0.0025,
        0.0249, 0.0055, 0.0339, 0.0109, 0.0029, 0.0198, 0.0118, 0.1537, 0.1459])
label (actual next character): 5
probability assigned by the net to the correct character: 0.01228625513613224
log likelihood: -4.399273872375488
negative log likelihood: 4.399273872375488
==========
bigram example 2: em (indexes 5, 13)
input to the neural net: 5
output probabilities from the neural net: 
tensor([0.0290, 0.0796, 0.0248, 0.0521, 0.1989, 0.0289, 0.0094, 0.0335, 0.0097,
        0.0301, 0.0702, 0.0228, 0.0115, 0.0181, 0.0108, 0.0315, 0.0291, 0.0045,
        0.0916, 0.0215, 0.0486, 0.0300, 0.0501, 0.0027, 0.0118, 0.0022, 0.0472])
label (actual next character): 13
probability assigned by the net to the correct character: 0.018050700426101685
log likelihood: -4.014570713043213
negative log likelihood: 4.014570713043213
...
...
...
==========
bigram example 5: a. (indexes 1, 0)
input to the neural net: 1
output probabilities from the neural net: 
tensor([0.0150, 0.0086, 0.0396, 0.0100, 0.0606, 0.0308, 0.1084, 0.0131, 0.0125,
        0.0048, 0.1024, 0.0086, 0.0988, 0.0112, 0.0232, 0.0207, 0.0408, 0.0078,
        0.0899, 0.0531, 0.0463, 0.0309, 0.0051, 0.0329, 0.0654, 0.0503, 0.0091])
label (actual next character): 0
probability assigned by the net to the correct character: 0.014977526850998402
log likelihood: -4.201204299926758
negative log likelihood: 4.201204299926758
==========
average negative lot likelihood, i.e. loss = 3.7693049907684326
```
</small>

<h1> Running Gradient Descent on "Emma" </h1>

Training set of bigrams: 
<small>
```python
# Creating training set of bigrams (x, y) for "Emma" 
xs, ys = [], [] 
for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1] ; ix2 = stoi[ch2]
        xs.append(ix1) ; ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
# Randomly initialize 27 neurons' weights. Each neuron receives 27 inputs 
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator = g, requires_grad = True)
```
</small>

Output:
<small>
```
tensor([ 0,  5, 13, 13,  1])
tensor([ 5, 13, 13,  1,  0])
```
</small>

Running Gradient Descent: 
<small>
```python
# Gradient Descent: 
for k in range(10):
    # Forward pass: 
    xenc = F.one_hot(xs, num_classes = 27).float() # input to the network: one-hot encoding 
    logits = xenc @ W # predict log-counts 
    counts = logits.exp() # counts, equivalent to N 
    probs = counts / counts.sum(1, keepdims = True) # probabilities for next character 
    loss = -probs[torch.arange(5), ys].log().mean()
    print(loss.item())
    # Backward pass: 
    W.grad = None # Set the gradients to 0
    loss.backward() 
    # Update gradients: 
    W.data += -0.1 * W.grad 
```
</small>

Output: 
<small>
```
3.6692662239074707
3.6493873596191406
3.629552125930786
3.6097614765167236
3.5900158882141113
3.5703155994415283
3.5506606101989746
3.5310521125793457
3.5114905834198
3.491975784301758
```
</small>

In [ ]:
# Creating training set of bigrams (x, y) for "Emma" 
xs, ys = [], [] 
for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1] ; ix2 = stoi[ch2]
        xs.append(ix1) ; ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
# Randomly initialize 27 neurons' weights. Each neuron receives 27 inputs 
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator = g, requires_grad = True)

tensor([ 0,  5, 13, 13,  1])
tensor([ 5, 13, 13,  1,  0])


In [ ]:
# Gradient Descent: 
for k in range(10):
    # Forward pass: 
    xenc = F.one_hot(xs, num_classes = 27).float() # input to the network: one-hot encoding 
    logits = xenc @ W # predict log-counts 
    counts = logits.exp() # counts, equivalent to N 
    probs = counts / counts.sum(1, keepdims = True) # probabilities for next character 
    loss = -probs[torch.arange(5), ys].log().mean()
    print(loss.item())
    # Backward pass: 
    W.grad = None # Set the gradients to 0
    loss.backward() 
    # Update gradients: 
    W.data += -0.1 * W.grad 

3.6692662239074707
3.6493873596191406
3.629552125930786
3.6097614765167236
3.5900158882141113
3.5703155994415283
3.5506606101989746
3.5310521125793457
3.5114905834198
3.491975784301758
